# Two-Step and One-Step categorisation
Leakage-free, checkpointed rebuild of the five Two-Step levels (general, computer security, hardware, operating systems, software) in one parametrised notebook: protocol in `notebooks/AGENTS.md` and `notebooks/rebuild.md`, values in `notebooks/categorisation.toml`, logic in `notebooks/categorisation.py`. The level is chosen with `CATEGORISATION_LEVEL`; the sixth level `one step` is the One-Step top level (all corpus rows, irrelevant and relevant-unmapped posts labelled `other`) and additionally reports the implicit relevance (any non-`other` label predicted) against the gold relevance. Per (embedder, method x classifier) candidate: one out-of-fold prediction per eligible train row from the manifest inner folds; per embedder each label takes the candidate with the best pooled OOF F1; the embedder with the best composed OOF macro F1 wins; the selected candidates are refit on all eligible train rows and evaluated once on test with group-bootstrap CIs. Reads only the shared split and the embedding artifacts (it embeds nothing).

In [ ]:
import hashlib
import json
import os
import platform
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
os.chdir(os.getenv("ROOT"))

from notebooks import categorisation as ct
from notebooks import checkpoints as ck
from notebooks import embeddings, splits

In [ ]:
def env_flag(name):
    return os.getenv(name, "").strip().lower() in ("1", "true", "yes")


SMOKE = env_flag("CATEGORISATION_SMOKE")  # tiny NON-PRODUCTION check; never writes the production outputs
OVERWRITE = env_flag("OVERWRITE")
ALLOW_UNRESOLVED = env_flag("ALLOW_UNRESOLVED_EMBEDDERS")
RANDOM_STATE = int(os.environ["RANDOM_STATE"])
FROZEN_SEED = 42
STAGE = os.getenv("STAGE", "all").strip().lower()
if STAGE not in ("train", "assemble", "all"):
    raise ValueError(f"STAGE must be train, assemble or all, got {STAGE!r}")
TRAIN, ASSEMBLE = STAGE in ("train", "all"), STAGE in ("assemble", "all")
SPLIT_MANIFEST_CSV = Path(os.environ["SPLIT_MANIFEST_CSV"])
SPLIT_TRAIN_CSV = Path(os.environ["SPLIT_TRAIN_CSV"])
SPLIT_TEST_CSV = Path(os.environ["SPLIT_TEST_CSV"])
SPLIT_MANIFEST_SHA256 = os.environ["SPLIT_MANIFEST_SHA256"]
EMBEDDINGS_DIR = Path(os.environ["EMBEDDINGS_DIR"])

CFG = ct.effective_config(ct.load_config("notebooks/categorisation.toml"), SMOKE)
LEVEL = os.getenv("CATEGORISATION_LEVEL", "").strip()
if LEVEL not in CFG["selection"]["levels"] or LEVEL not in CFG["levels"]:
    raise ValueError(f"CATEGORISATION_LEVEL must be one of {CFG['selection']['levels']}, got {LEVEL!r}")
SLUG = re.sub(r"[^a-z0-9]+", "_", LEVEL.lower()).strip("_")
EMBEDDER_NAMES = list(CFG["selection"]["embedders"])
SHARD = [e.strip() for e in os.getenv("EMBEDDERS", "").split(",") if e.strip()]
if SHARD:
    if STAGE != "train":
        raise ValueError("EMBEDDERS (a shard subset) is only allowed with STAGE=train; assemble covers the full configuration")
    unknown = [e for e in SHARD if e not in EMBEDDER_NAMES]
    if unknown:
        raise ValueError(f"EMBEDDERS {unknown} are not configured embedders {EMBEDDER_NAMES}")
    SHARD = [e for e in EMBEDDER_NAMES if e in SHARD]
else:
    SHARD = list(EMBEDDER_NAMES)

# Outputs: CATEGORISATION_OUT_DIR (default models/); smoke writes only under CATEGORISATION_SMOKE_OUT_DIR.
PROD_DIR = Path(os.getenv("CATEGORISATION_OUT_DIR", "models"))
PRODUCTION = {k: PROD_DIR / n for k, n in (("pickle", f"categoriser_{SLUG}.pkl"),
                                           ("csv", f"categorisation_{SLUG}_performance.csv"),
                                           ("json", f"categorisation_{SLUG}_performance.json"))}
if SMOKE:
    OUT_DIR = Path(os.environ["CATEGORISATION_SMOKE_OUT_DIR"])
    OUT = {k: OUT_DIR / f"smoke_{p.name}" for k, p in PRODUCTION.items()}
    for k in OUT:
        ct.refuse_production_output(OUT[k], PRODUCTION[k])
else:
    OUT = PRODUCTION
if ASSEMBLE and not OVERWRITE:
    existing = [str(p) for p in (*OUT.values(), Path(str(OUT["pickle"]) + ".json")) if p.exists()]
    if existing:
        raise FileExistsError(f"{existing} exist; set OVERWRITE=true to replace them")

CANDS = ct.build_candidates(CFG)
BOOT = CFG["bootstrap"]
print(f"{'NON-PRODUCTION SMOKE' if SMOKE else 'PRODUCTION'}: level {LEVEL!r}, stage {STAGE}, {len(CANDS)} candidates, "
      f"embedders {SHARD if TRAIN else EMBEDDER_NAMES}")

In [ ]:
# Data: split files verified against the pinned manifest; labels and population from the taxonomy (gold labels only).
train_df = pd.read_csv(SPLIT_TRAIN_CSV)
test_df = pd.read_csv(SPLIT_TEST_CSV)
manifest = splits.read_manifest(SPLIT_MANIFEST_CSV)
full = pd.concat([train_df, test_df], ignore_index=True)
splits.verify_manifest(full, manifest, expected_seed=FROZEN_SEED, expected_manifest_sha256=SPLIT_MANIFEST_SHA256)
COLS = manifest.attrs["metadata"]["columns"]
MANIFEST_IDS = manifest["row_id"].astype(str).tolist()
DATA = ct.prepare_level(train_df, test_df, manifest, LEVEL, CFG, columns=COLS)
MIN_COUNT = int(CFG["selection"]["min_labelset_count"])
if not DATA.train_ids or not DATA.test_ids:
    raise ct.CategorisationError(f"level {LEVEL!r}: no eligible rows (train {len(DATA.train_ids)}, test {len(DATA.test_ids)})")
try:
    PLANS = ct.plan_folds(DATA, MIN_COUNT)  # fails closed when a fold has no frequent labelset (sparse level)
except ct.CategorisationError as e:
    raise ct.CategorisationError(f"SPARSE LEVEL {LEVEL!r} cannot be run with the >= {MIN_COUNT} labelset rule: {e}") from e
print(f"manifest verified ({SPLIT_MANIFEST_SHA256[:12]}...): level {LEVEL!r} classes {list(DATA.classes)}; "
      f"eligible train {len(DATA.train_ids)}, test {len(DATA.test_ids)}; fitting rows per fold "
      f"{[len(p.fit_idx) for p in PLANS]}; train class support {DATA.Y_train.sum(axis=0).tolist()}")

In [ ]:
# Embedding artifacts bound to the manifest row ids (an 'unresolved' fingerprint fails closed unless ALLOW_UNRESOLVED_EMBEDDERS=1).
ARTS = {}
for name in (SHARD if STAGE == "train" else EMBEDDER_NAMES):
    ARTS[name] = embeddings.load_artifact(EMBEDDINGS_DIR, embeddings.artifact_name("corpus", name),
                                          expected_row_ids=MANIFEST_IDS, allow_unresolved=ALLOW_UNRESOLVED)


def features(embedder, ids):
    return ARTS[embedder].select(ids)  # test rows are selected only by evaluate_test, after every choice


print({n: (a.shape, a.fingerprint.get("revision", a.fingerprint.get("sha256", "")[:12])) for n, a in ARTS.items()})

In [ ]:
# Checkpoint identity: code, config, level, split, embedding sidecars of ALL configured embedders (so that
# every shard of a level shares one identity) and library versions. The shard subset is not part of it.
CODE = {n: f"notebooks/{n}" for n in ("categorisation.py", "categorisation.toml", "filtering.py", "textprep.py", "embeddings.py",
                                      "splits.py", "taxonomy.py", "taxonomy.toml", "checkpoints.py")}
CODE.update({f"MLC/{p.name}": str(p) for p in sorted(Path("MLC").glob("*.py"))})
DATA_PATHS = {"manifest": SPLIT_MANIFEST_CSV, "train": SPLIT_TRAIN_CSV, "test": SPLIT_TEST_CSV}
DATA_PATHS.update({f"embedding-{n}": EMBEDDINGS_DIR / f"{embeddings.artifact_name('corpus', n)}.json" for n in EMBEDDER_NAMES})
CONFIG_DIGEST = hashlib.sha256(json.dumps(CFG, sort_keys=True, default=str).encode()).hexdigest()
IDENTITY = ck.make_identity(
    code_paths=CODE, data_paths=DATA_PATHS,
    modules=["numpy", "pandas", "scikit-learn", "scipy", "joblib"],
    config={"categorisation": CFG, "level": LEVEL, "seed": RANDOM_STATE, "manifest_sha256": SPLIT_MANIFEST_SHA256})
STORE = ck.Store.from_env(IDENTITY)
BUDGET = ck.Budget.from_env()
STOPPED = False
SHARD_KEYS = ct.expected_keys(DATA, SHARD, CANDS)
ALL_KEYS = ct.expected_keys(DATA, EMBEDDER_NAMES, CANDS)
OOF = {}
print("identity", IDENTITY["digest"][:12], "| store", STORE.out_dir)

## Out-of-fold predictions of every candidate (train only)

In [ ]:
def show_failures(oof):
    bad = {(e, c): r["reason"] for e, recs in oof.items() for c, r in recs.items() if r["status"] == ct.FAILED}
    for (e, c), why in bad.items():
        print(f"FAILED {LEVEL!r} / {e} / {c}: {why}")
    return bad


if TRAIN:
    for emb in SHARD:
        run = ct.run_oof(DATA, emb, CANDS, CFG, features, DATA.test_ids, seed=RANDOM_STATE, store=STORE, budget=BUDGET)
        OOF[emb] = run.records
        if not run.complete:
            STOPPED = True
            missing = [k for k in SHARD_KEYS if not STORE.has(k)]
            ck.write_status(STORE.out_dir, "INCOMPLETE", done=[k for k in SHARD_KEYS if k not in missing], missing=missing,
                            identity_digest=STORE.digest)
            print(f"INCOMPLETE: {len(SHARD_KEYS) - len(missing)}/{len(SHARD_KEYS)} fold checkpoints within the budget; rerun to resume.")
            break
        n_ok = sum(r["status"] == ct.OK for r in run.records.values())
        print(f"{LEVEL!r} / {emb}: {n_ok}/{len(CANDS)} candidates scored OOF, "
              f"{sum(r['status'] == ct.INFEASIBLE for r in run.records.values())} infeasible by design")
    show_failures(OOF)
    if not STOPPED and STAGE == "train":
        ck.verify_complete(STORE, SHARD_KEYS)
        ck.write_status(STORE.out_dir, "COMPLETE", done=SHARD_KEYS, identity_digest=STORE.digest)
        print("COMPLETE train shard", SHARD, "(NON-PRODUCTION smoke)" if SMOKE else "")

## Selection (OOF selection scores), refit and one test evaluation

In [ ]:
if ASSEMBLE and not STOPPED:
    ck.verify_complete(STORE, ALL_KEYS)  # every (embedder, candidate, fold) of the level, from the shards' stores
    for emb in EMBEDDER_NAMES:
        if emb not in OOF:  # assemble reads the checkpoints only; the zero budget forbids any fit
            run = ct.run_oof(DATA, emb, CANDS, CFG, features, DATA.test_ids, seed=RANDOM_STATE, store=STORE,
                             budget=ck.Budget(hours=0))
            assert run.complete, f"{emb}: checkpoints incomplete"
            OOF[emb] = run.records
    if show_failures(OOF):
        raise ct.CategorisationError(f"{LEVEL!r}: candidates failed to fit (see FAILED lines); fix the cause and rerun the shard; "
                                     "failed records are never selected around in production")
    SEL = ct.select(DATA, OOF, [c.name for c in CANDS], EMBEDDER_NAMES, forbidden_ids=DATA.test_ids, allow_failed=False)
    TABLE = SEL.table()
    print(f"OOF SELECTION SCORES, level {LEVEL!r} ({ct.SELECTION_NOTE}); not performance estimates:")
    display(TABLE)
    print("composed OOF macro F1 per embedder:", {e: round(v, 4) for e, v in SEL.embedder_scores.items()})
    print("selected:", SEL.embedder, dict(zip(SEL.classes, SEL.label_candidates)))
    print("excluded (infeasible by design):", {e: len(v) for e, v in SEL.excluded.items()})
    CAT = ct.refit(DATA, SEL, CFG, features, DATA.test_ids, seed=RANDOM_STATE)
    TEST = ct.evaluate_test(CAT, DATA, CFG, features, B=int(BOOT["B"]), seed=int(BOOT["seed"]), level=float(BOOT["level"]))
    for scope in ("all", "thesis_comparable"):
        t = TEST[scope]
        if t["n"] == 0:
            print(f"TEST [{scope}]: no rows")
            continue
        ci = t["bootstrap"]["ci"]
        print(f"TEST [{scope}] n={t['n']} (primary: all): " + ", ".join(
            f"{k} {t[k]:.3f} [{ci[k][0]:.3f}, {ci[k][1]:.3f}]" for k in ("micro_f1", "macro_f1", "weighted_f1", "subset_accuracy", "hamming_loss")))
        print({c: round(v["f1"], 3) for c, v in t["labels"].items()})
        r = t.get("relevance")
        if r:  # One-Step (population all): implicit relevance = any non-'other' label predicted
            rci = r.get("bootstrap", {}).get("ci", {})
            print(f"RELEVANCE [{scope}] n={r['n']} prevalence {r['prevalence']:.3f} confusion [[tn, fp], [fn, tp]] {r['confusion_matrix']}, empty predictions {r['empty_predictions']}, other+real {r['other_with_real']}: " + ", ".join(
                f"{k} {r[k]:.3f}" + (f" [{rci[k][0]:.3f}, {rci[k][1]:.3f}]" if k in rci else "")
                for k in ("precision", "recall", "f1", "accuracy", "balanced_accuracy", "roc_auc")))
            print(f"RELEVANCE [{scope}] relevant-unmapped n={r['relevant_unmapped']['n']} recall {r['relevant_unmapped']['recall']:.3f}; "
                  f"relevant-mapped n={r['relevant_mapped']['n']} recall {r['relevant_mapped']['recall']:.3f}")

## Save the categoriser, performance tables and sidecar

In [ ]:
if ASSEMBLE and not STOPPED:
    def lib_versions():
        import importlib.metadata as md
        return {p: md.version(p) for p in ("numpy", "pandas", "scikit-learn", "scipy", "joblib")}

    rows = []
    for scope in ("all", "thesis_comparable"):
        t = TEST[scope]
        if t["n"] == 0:
            continue
        lci = t["bootstrap"]["label_f1_ci"]
        for c, v in t["labels"].items():
            rows.append({"level": LEVEL, "scope": scope, "label": c, "candidate": CAT.label_candidates[DATA.classes.index(c)],
                         "n": t["n"], "support": v["support"], "precision": v["precision"], "recall": v["recall"],
                         "f1": v["f1"], "f1_ci_low": lci[c][0], "f1_ci_high": lci[c][1], "roc_auc": v["roc_auc"]})
        rows.append({"level": LEVEL, "scope": scope, "label": "__overall__", "candidate": "", "n": t["n"],
                     **{k: t[k] for k in ("micro_f1", "macro_f1", "weighted_f1", "subset_accuracy", "hamming_loss", "macro_roc_auc")}})
        if "relevance" in t:  # One-Step: implicit relevance vs gold relevance (also stored in the json and sidecar)
            r = t["relevance"]
            rows.append({"level": LEVEL, "scope": scope, "label": "__relevance__", "candidate": "", "n": r["n"], "support": int(r["confusion_matrix"][1][0] + r["confusion_matrix"][1][1]),
                         "precision": r["precision"], "recall": r["recall"], "f1": r["f1"], "roc_auc": r["roc_auc"]})
    PERF = pd.DataFrame(rows)
    Path(OUT["pickle"]).parent.mkdir(parents=True, exist_ok=True)
    METRICS = {"test": TEST, "oof_selection_scores": {"note": ct.SELECTION_NOTE, "embedder_scores": SEL.embedder_scores,
                                                      "rows": list(SEL.rows)},
               "bootstrap": {**BOOT, "unit": "manifest group", "conditional_on": "the selected pipeline"}}
    SIDECAR = {
        "level": LEVEL, "non_production": SMOKE,
        "embedder": {"name": SEL.embedder, "fingerprint": ARTS[SEL.embedder].fingerprint},
        "classes": list(DATA.classes), "label_candidates": list(SEL.label_candidates),
        "manifest_sha256": SPLIT_MANIFEST_SHA256, "random_state": RANDOM_STATE,
        "config_digest": CONFIG_DIGEST, "identity_digest": IDENTITY["digest"],
        "refit": CAT.info, "metrics": METRICS,
        "metric_note": "test metrics are the estimate (primary scope 'all'); OOF scores are optimistic selection scores",
        "library_versions": lib_versions(), "python": platform.python_version(),
    }
    SIDECAR["pickle_sha256"] = ct.stage_categoriser(CAT, OUT["pickle"])
    PERF.to_csv(ct.staged(OUT["csv"]), index=False)
    Path(ct.staged(OUT["json"])).write_text(json.dumps(METRICS, indent=1, sort_keys=True, default=str))
    DOC = ct.publish_categoriser(OUT["pickle"], SIDECAR, extra_paths=[OUT["csv"], OUT["json"]], smoke=SMOKE,
                                 production_paths=list(PRODUCTION.values()))
    print("saved", OUT["pickle"], "+ .json sidecar,", OUT["csv"], OUT["json"])
    ck.write_status(STORE.out_dir, "COMPLETE", done=ALL_KEYS, identity_digest=STORE.digest)
    print("COMPLETE", f"level {LEVEL!r}", "(NON-PRODUCTION smoke)" if SMOKE else "")